# 线性回归与逻辑回归有什么区别？

**面试回答主线：**线性回归预测连续条件均值，常以 MSE 为目标；逻辑回归以线性得分建模对数几率，再用 sigmoid 输出概率并最小化交叉熵。两者的线性指参数对特征的线性组合，不代表输出形式相同。本实验同时预测订单配送分钟数与是否超时。

## 真实案例

每条订单有配送距离和商家备餐分钟。业务既需要连续 ETA（回归），也需要提前识别是否超过 40 分钟（分类）。这两个目标相关但不能用同一损失和同一种输出直接替代。

In [1]:
import numpy as np  # 导入 NumPy 以手写两个线性模型。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的数值显示。
order = np.array(['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10'])  # 构造配送订单编号。
distance = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 2.5, 6.0, 3.5, 7.0, 4.5])  # 记录配送距离公里数。
prep = np.array([6, 8, 10, 15, 18, 9, 22, 12, 25, 16], dtype=float)  # 记录商家备餐分钟。
eta = np.array([18, 23, 29, 38, 45, 26, 55, 34, 64, 42], dtype=float)  # 记录实际配送分钟数。
late = (eta > 40).astype(float)  # 从业务 SLA 派生是否超时的分类标签。
train_index = np.arange(7)  # 使用前七条历史订单训练。
valid_index = np.arange(7, 10)  # 使用后三条订单做未来回放。
print('订单 | 距离km | 备餐min | ETA分钟 | 是否超时')  # 输出业务数据表头。
for index in range(len(order)):  # 逐条展示订单特征和两个目标。
    print(f'{order[index]} | {distance[index]:4.1f} | {prep[index]:7.0f} | {eta[index]:7.0f} | {int(late[index])}')  # 输出一条订单记录。

订单 | 距离km | 备餐min | ETA分钟 | 是否超时
D01 |  1.0 |       6 |      18 | 0
D02 |  2.0 |       8 |      23 | 0
D03 |  3.0 |      10 |      29 | 0
D04 |  4.0 |      15 |      38 | 0
D05 |  5.0 |      18 |      45 | 1
D06 |  2.5 |       9 |      26 | 0
D07 |  6.0 |      22 |      55 | 1
D08 |  3.5 |      12 |      34 | 0
D09 |  7.0 |      25 |      64 | 1
D10 |  4.5 |      16 |      42 | 1


## Baseline / 基线

连续 ETA 的基线是历史平均分钟数；超时风险的基线是历史超时率。它们不使用订单特征，便于判断模型是否真的从距离和备餐时间学到信息。

In [2]:
eta_baseline = np.full(len(valid_index), eta[train_index].mean())  # 用训练集平均 ETA 预测所有未来订单。
late_baseline = np.full(len(valid_index), late[train_index].mean())  # 用训练集超时率作为所有未来订单风险。
eta_baseline_mse = float(np.mean((eta_baseline - eta[valid_index]) ** 2))  # 计算连续预测基线 MSE。
late_baseline_acc = float(np.mean((late_baseline >= 0.5) == late[valid_index]))  # 计算风险基线准确率。
print(f'ETA 均值基线 MSE={eta_baseline_mse:.2f}')  # 输出连续任务基线。
print(f'超时率基线 accuracy={late_baseline_acc:.3f}')  # 输出分类任务基线。

ETA 均值基线 MSE=336.14
超时率基线 accuracy=0.333


In [3]:
feature = np.c_[distance, prep]  # 拼接距离和备餐时间形成输入矩阵。
mean = feature[train_index].mean(axis=0)  # 仅在训练订单上计算特征均值。
std = feature[train_index].std(axis=0) + 1e-6  # 仅在训练订单上计算标准差并避免除零。
x_train = np.c_[np.ones(len(train_index)), (feature[train_index] - mean) / std]  # 构造包含截距的标准化训练特征。
x_valid = np.c_[np.ones(len(valid_index)), (feature[valid_index] - mean) / std]  # 使用训练统计量构造验证特征。
linear_weight = np.linalg.solve(x_train.T @ x_train + 0.01 * np.eye(x_train.shape[1]), x_train.T @ eta[train_index])  # 用带微弱稳定项的正规方程手写线性回归。
eta_probability_like = x_valid @ linear_weight  # 计算线性回归的连续 ETA 输出。
eta_mse = float(np.mean((eta_probability_like - eta[valid_index]) ** 2))  # 计算连续预测的验证 MSE。
print('线性回归权重 [截距, 距离, 备餐]:', np.round(linear_weight, 3))  # 展示 ETA 模型参数。
print('线性回归 ETA:', np.round(eta_probability_like, 2))  # 展示连续输出并强调它不是概率。
print(f'线性回归验证 MSE={eta_mse:.2f}')  # 输出回归指标。

线性回归权重 [截距, 距离, 备餐]: [33.381  3.988  8.22 ]
线性回归 ETA: [32.87 61.11 41.37]
线性回归验证 MSE=3.35


In [4]:
logistic_weight = np.zeros(x_train.shape[1])  # 初始化超时分类的逻辑回归权重。
for step in range(300):  # 迭代执行二分类交叉熵的梯度下降。
    train_probability = 1.0 / (1.0 + np.exp(-(x_train @ logistic_weight)))  # 计算训练订单的超时概率。
    gradient = x_train.T @ (train_probability - late[train_index]) / len(train_index)  # 计算伯努利负对数似然的梯度。
    logistic_weight -= 0.25 * gradient  # 更新风险分类权重。
late_probability = 1.0 / (1.0 + np.exp(-(x_valid @ logistic_weight)))  # 将线性风险得分映射为零到一概率。
late_prediction = (late_probability >= 0.5).astype(float)  # 按阈值生成是否超时的业务决策。
late_accuracy = float(np.mean(late_prediction == late[valid_index]))  # 计算分类验证准确率。
print('逻辑回归权重 [截距, 距离, 备餐]:', np.round(logistic_weight, 3))  # 展示分类模型参数。
print('逻辑回归超时概率:', np.round(late_probability, 3))  # 展示可以排序和阈值化的风险概率。
print(f'逻辑回归验证 accuracy={late_accuracy:.3f}')  # 输出分类指标。

逻辑回归权重 [截距, 距离, 备餐]: [-2.907  2.255  2.263]
逻辑回归超时概率: [0.05  0.999 0.527]
逻辑回归验证 accuracy=1.000


## 结果解读

线性回归给出分钟数，可用 MSE/MAE 评估；逻辑回归给出超时概率，可用 ROC、PR、校准和业务成本阈值评估。将线性回归的分钟数硬截到 0 到 1 不是合法概率建模，反过来把逻辑概率当 ETA 也没有单位含义。

In [5]:
print('订单 | 真实ETA | 线性ETA | 真实超时 | 逻辑概率 | 分类')  # 输出两个任务的逐订单结果表头。
for local_index, global_index in enumerate(valid_index):  # 逐条展示回归和分类输出。
    print(f'{order[global_index]} | {eta[global_index]:7.1f} | {eta_probability_like[local_index]:7.1f} | {int(late[global_index])}        | {late_probability[local_index]:7.3f} | {int(late_prediction[local_index])}')  # 输出同一订单的两个目标。
print('教学结论：同一线性特征可以服务不同任务，但概率、损失和指标必须与业务目标对应。')  # 总结两类回归的差异。

订单 | 真实ETA | 线性ETA | 真实超时 | 逻辑概率 | 分类
D08 |    34.0 |    32.9 | 0        |   0.050 | 0
D09 |    64.0 |    61.1 | 1        |   0.999 | 1
D10 |    42.0 |    41.4 | 1        |   0.527 | 1
教学结论：同一线性特征可以服务不同任务，但概率、损失和指标必须与业务目标对应。


## 失败案例与修复

失败做法：直接把线性回归输出当作超时概率。它可能为负或大于 1。修复是使用 sigmoid 和交叉熵得到受约束概率，再在验证集按漏报/误报成本选阈值。

In [6]:
wrong_probability = eta_probability_like / 40.0  # 错误地把 ETA 除以 SLA 伪装成概率。
wrong_probability = np.clip(wrong_probability, 0.0, 1.0)  # 临时裁剪只能遮盖而不能修复错误建模。
threshold = 0.35  # 假设漏掉超时的成本高于多一次人工复核。
cost_sensitive_prediction = (late_probability >= threshold).astype(float)  # 在验证概率上应用业务阈值。
print('错误 ETA/40 概率:', np.round(wrong_probability, 3))  # 展示无统计含义的伪概率。
print('修复逻辑概率:', np.round(late_probability, 3))  # 展示伯努利模型输出的概率。
print(f'阈值 {threshold:.2f} 下的人工复核订单数={int(cost_sensitive_prediction.sum())}')  # 展示阈值如何连接业务动作。
print('生产差距：需补充置信区间、概率校准、分城市 SLA、延迟监控和延迟标签回灌。')  # 说明实际服务还需的能力。

错误 ETA/40 概率: [0.822 1.    1.   ]
修复逻辑概率: [0.05  0.999 0.527]
阈值 0.35 下的人工复核订单数=2
生产差距：需补充置信区间、概率校准、分城市 SLA、延迟监控和延迟标签回灌。


In [7]:
assert len(order) >= 5  # 保护案例包含至少五条订单。
assert eta_mse < eta_baseline_mse  # 保护连续模型在教学回放中优于均值基线。
assert np.all((late_probability > 0.0) & (late_probability < 1.0))  # 保护逻辑回归输出是真实概率范围。
assert np.any(eta_probability_like > 40.0)  # 保护线性回归输出具有分钟单位而不是概率。